In [67]:
import os
import sys
import pandas as pd
from ydata_profiling import ProfileReport
current = os.getcwd()
path_to_root = os.path.join (current, '../')
abs_path = os.path.abspath(path_to_root)
sys.path.append(abs_path)
import config
import ijson
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

client = config.create_minio_client()

[Bucket('aws'), Bucket('azure'), Bucket('azure-clean'), Bucket('google'), Bucket('google-clean')]


In [68]:
# object_name = "AmazonEC2.json"
# object_name = "AmazonTimestream.json"
object_name = "AmazonS3.json"

TARGET_RECORDS = 100000 
sample_products = []

response = client.get_object(config.PROVIDERS.get("aws").get("bucket"), object_name=object_name)

try:
    # Το ijson διαβάζει κατευθείαν από το stream byte-byte
    parser = ijson.kvitems(response, 'products')
    
    count = 0
    for sku, product_data in parser:
        #Mε την προοπτική να δημιουργεί πεδίο με sku με την αντίστοιχη τιμή μέσα στο dict αλλά αυτό ήδη υπάρχει
        # product_data['sku'] = sku
        sample_products.append(product_data)
        
        count += 1
        if count >= TARGET_RECORDS:
            break
            
    print(f"Downloaded  {len(sample_products)} records μέσω streaming.")

finally:
    response.close()
    response.release_conn()

# Μετατροπή σε αρχικό DataFrame
df_products = pd.json_normalize(sample_products)
print(f"DataFrame: Rows = {df_products.shape[0]}, Columns = {df_products.shape[1]}")
df_products.head()

Downloaded  9031 records μέσω streaming.
DataFrame: Rows = 9031, Columns = 25


,sku,attributes.servicecode,attributes.transferType,attributes.fromLocation,attributes.fromLocationType,attributes.toLocation,attributes.toLocationType,attributes.usagetype,attributes.operation,attributes.fromRegionCode,attributes.servicename,attributes.toRegionCode,productFamily,attributes.location,attributes.locationType,attributes.availability,attributes.storageClass,attributes.volumeType,attributes.durability,attributes.regionCode,attributes.feeCode,attributes.feeDescription,attributes.group,attributes.groupDescription,attributes.overhead
0,UBHUUCQA2BAWFNFA,AmazonS3,IntraRegion Outbound,Asia Pacific (Osaka),AWS Region,Asia Pacific (Osaka),AWS Region,APN3-APN3-S3RTC-Out-Bytes,,ap-northeast-3,Amazon Simple Storage Service,ap-northeast-3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5PP56XDP5UX3QP8N,AmazonS3,InterRegion Inbound,Canada West (Calgary),AWS Region,Asia Pacific (Melbourne),AWS Region,APS6-CAN2-S3RTC-In-Bytes,,ca-west-1,Amazon Simple Storage Service,ap-southeast-4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7H35Q77CGKE5UBSJ,AmazonS3,NaN,NaN,NaN,NaN,NaN,CAN2-TimedStorage-ByteHrs,,NaN,Amazon Simple Storage Service,NaN,Storage,Canada West (Calgary),AWS Region,99.99%,General Purpose,Standard,99.999999999%,ca-west-1,NaN,NaN,NaN,NaN,NaN
3,EXBKATS7UXZKUK4R,AmazonS3,InterRegion Inbound,Asia Pacific (Melbourne),AWS Region,Asia Pacific (Hyderabad),AWS Region,APS5-APS6-S3RTC-In-Bytes,,ap-southeast-4,Amazon Simple Storage Service,ap-south-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,YD4SUKU86M9QUVVW,AmazonS3,NaN,NaN,NaN,NaN,NaN,APS2-Tables-SortProcessedBytes,,NaN,Amazon Simple Storage Service,NaN,NaN,Asia Pacific (Sydney),AWS Region,NaN,NaN,NaN,NaN,ap-southeast-2,S3-Tables-Sort-ProcessedBytes,Fee for bytes processed for sort or Z-order co...,NaN,NaN,NaN


In [69]:
# Στο πάνω κελί είχα μία λίστα η οποία είχε μέσα n sku, μαζί με όλα τα attributes τουσ.
# Τώρα στο βήμα αυτό κάνω access την λίστα και απομονώνω σε ένα set μόνο τον κωδικό των n sku (η επιλογή set βασίζεται στην γρήγορη αναζήτηση)
target_skus = {p['sku'] for p in sample_products}

# Αυτή θα είναι η αντίστοιχη sample products του πάνω βήματος. Θα κρατάει τα ζευγάρια sku, με στοιχεία πληρωμής, και μετά θα την κάνουμε dataframe
terms_list = []

parser = client.get_object(config.PROVIDERS.get("aws").get("bucket"), object_name=object_name)

try:
    # Πάμε βαθύτερα και από το λεξικό terms θα στοχεύσουμε μόνο στις On-Demand υπηρεσίες.
    terms_parser = ijson.kvitems(parser, 'terms.OnDemand')

    # Προφανώς δεν τα θέλω όλα!!Μόνο εκείνα των οποίων το sku βρίσκεται στο set το οποίο δημιούργησα
    for sku, term_offers in terms_parser:
        if sku not in target_skus:
            continue
        
        # Αποθηκεύουμε το sku και ολόκληρο το raw λεξικό των terms του
        terms_list.append({
            'skuNew': sku,
            'termsOndemand': term_offers
        })
        
        # Aπλά για να βεβαιωθώ ότι όσα products πήρα άλλες τόσες και οι τιμές 
        if len(terms_list) >= len(target_skus):
            break
            
    print(f"Downloaded {len(terms_list)} matching terms μέσω streaming.")

finally:
    parser.close()
    parser.release_conn()


df_terms = pd.DataFrame(terms_list)
print(f"Terms DataFrame: Rows = {df_terms.shape[0]}, Columns = {df_terms.shape[1]}")
df_terms.head()

Downloaded 9031 matching terms μέσω streaming.
Terms DataFrame: Rows = 9031, Columns = 2


,skuNew,termsOndemand
0,UBHUUCQA2BAWFNFA,{'UBHUUCQA2BAWFNFA.JRTCKXETXF': {'offerTermCod...
1,5PP56XDP5UX3QP8N,{'5PP56XDP5UX3QP8N.JRTCKXETXF': {'offerTermCod...
2,7H35Q77CGKE5UBSJ,{'7H35Q77CGKE5UBSJ.JRTCKXETXF': {'offerTermCod...
3,EXBKATS7UXZKUK4R,{'EXBKATS7UXZKUK4R.JRTCKXETXF': {'offerTermCod...
4,YD4SUKU86M9QUVVW,{'YD4SUKU86M9QUVVW.JRTCKXETXF': {'offerTermCod...


Θα δουλέψουμε αρχικά με το 2ο dataframe το οποίο περιέχει τα δεδομένα τιμολόγησης. Κρίνονται απαραίτητες 4 ενέργειες
- Άνοιγμα 2η στήλης και άπλωμα δεδομένων
- Αντιστοιχία εσωτερικού sku με αυτό που έβαλα εγώ, και πέταμα μίας στήλης εκ των 2
- Μελέτη για εντοπισμό καθολικών στηλών 
- Αφαίρεση περιττών στηλών
- Κατανόηση pricing και αντιστοίχιση με τους άλλους παρόχους

In [70]:
# Πάιρνω την πρώτη εγγραφή προκειμένου να κάνω έναν έλεγχο των πεδίων
sample_row = df_terms.iloc[0]
print("SKU:", sample_row['skuNew'])

raw_dict = sample_row['termsOndemand']

# Βρίσκουμε το κλειδί (το σύνθετο hash, π.χ. SKU.OfferTermCode)
offer_hash_key = list(raw_dict.keys())[0]
offer_content = raw_dict[offer_hash_key]

print("\n--- Περιεχόμενα προσφοράς (Offer Content) ---")
for k, v in offer_content.items():
    # if k != 'priceDimensions':
    print(f"{k}: {v}")

SKU: UBHUUCQA2BAWFNFA

--- Περιεχόμενα προσφοράς (Offer Content) ---
offerTermCode: JRTCKXETXF
sku: UBHUUCQA2BAWFNFA
effectiveDate: 2026-06-01T00:00:00Z
priceDimensions: {'UBHUUCQA2BAWFNFA.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'UBHUUCQA2BAWFNFA.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.015 per GB - Asia Pacific (Osaka) Data Transfer for Replication Time Control to Asia Pacific (Osaka)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'GB', 'pricePerUnit': {'USD': '0.0150000000'}, 'appliesTo': []}}
termAttributes: {}


Ο παρακάτω κώδικας υλοποιεί ένα μέρος του πρώτου βήματος. Πετάει ένα περιττό hash το οποίο ήταν sku + offerCode, και ανοίγει εν μέρη το λεξικό termsOnDemand το οποίο κατασκεύσαμε όταν πήραμε τα δεδομένα και τα μετατρέψαμε σε datframe. Συγκεκριμένα εξάγει μερικά πεδία και τα κάνει κανονικές στήλες. Το απευθείας normalize δοκιμάστηκε και απετύχε, οπότε και προχωρήσαμε με ένα for loop.

In [71]:
flattened_list = []

for idx, row in df_terms.iterrows():
    skuDefaultValue = row['skuNew']
    raw_dict = row['termsOndemand']
    
    # ΜΠετάω το αρχικό κλειδί το οποίο είχε το dict. Περισσοτερα στην αναφορά
    for hash_key, offer_content in raw_dict.items():
        
        # Εξάγω τα πεδία ένα - ένα και φτιάχνω μία δική μου δομή πιο υύκολη στην ανάλυση
        item = {
            'skuNew': skuDefaultValue,
            'offerTermCode': offer_content.get('offerTermCode'),
            'sku': offer_content.get('sku'),
            'effectiveDate': offer_content.get('effectiveDate'),
            'termAttributes': offer_content.get('termAttributes'),
            'priceDimensions': offer_content.get('priceDimensions') # Το αφήνουμε λεξικό!
        }
        flattened_list.append(item)

df_terms = pd.DataFrame(flattened_list)

print(df_terms.columns)
df_terms.head(2)

Index(['skuNew', 'offerTermCode', 'sku', 'effectiveDate', 'termAttributes',
       'priceDimensions'],
      dtype='object')


,skuNew,offerTermCode,sku,effectiveDate,termAttributes,priceDimensions
0,UBHUUCQA2BAWFNFA,JRTCKXETXF,UBHUUCQA2BAWFNFA,2026-06-01T00:00:00Z,{},{'UBHUUCQA2BAWFNFA.JRTCKXETXF.6YS6EN2CT7': {'r...
1,5PP56XDP5UX3QP8N,JRTCKXETXF,5PP56XDP5UX3QP8N,2026-06-01T00:00:00Z,{},{'5PP56XDP5UX3QP8N.JRTCKXETXF.6YS6EN2CT7': {'r...


Επόμενο βήμα είναι το περαιτέρω άνοιγμα του λεξικού το οποίο κρύβει μέσα τις πληροφοριες τιμολόγησης. Συγκεκριμένα το priceDimensions. Όπως φαίνεται και από το πάνω αποτέλεσμα του κελιού, μέσα στο λεξικό αυτό υπάρχει άλλο ένα περίεργο hash - κλειδί, το οποίο όμως όπως και στο παραπάνω κελί θα απορρίψουμε. Ο κώδικας λοιπόν ακολουθεί την ίδια τακτική: διατρέχει γραμμή γραμμή, προσπερνάει το περίεργο αυτό, και τραβάει μόνο τα πραγματικά δεδομένα.

In [72]:
df_terms.head()

,skuNew,offerTermCode,sku,effectiveDate,termAttributes,priceDimensions
0,UBHUUCQA2BAWFNFA,JRTCKXETXF,UBHUUCQA2BAWFNFA,2026-06-01T00:00:00Z,{},{'UBHUUCQA2BAWFNFA.JRTCKXETXF.6YS6EN2CT7': {'r...
1,5PP56XDP5UX3QP8N,JRTCKXETXF,5PP56XDP5UX3QP8N,2026-06-01T00:00:00Z,{},{'5PP56XDP5UX3QP8N.JRTCKXETXF.6YS6EN2CT7': {'r...
2,7H35Q77CGKE5UBSJ,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,{},{'7H35Q77CGKE5UBSJ.JRTCKXETXF.PGHJ3S3EYE': {'r...
3,EXBKATS7UXZKUK4R,JRTCKXETXF,EXBKATS7UXZKUK4R,2026-06-01T00:00:00Z,{},{'EXBKATS7UXZKUK4R.JRTCKXETXF.6YS6EN2CT7': {'r...
4,YD4SUKU86M9QUVVW,JRTCKXETXF,YD4SUKU86M9QUVVW,2026-06-01T00:00:00Z,{},{'YD4SUKU86M9QUVVW.JRTCKXETXF.6YS6EN2CT7': {'r...


In [73]:
itemsList = []

for idx, row in df_terms.iterrows():

    # Κατασκευάζουμε εξ'ολοκλήρου νεό dataframe. Δεν κάνουμε ενέργειες πάνω στον υπάρχον. Οπότε φτιάχνουμε γραμμή γραμμή με τα πεδία που έχουμε. Για αυτό και τα εξάξουμε ένα - ένα
    skuNew = row['skuNew']
    offerTermCode = row['offerTermCode']
    skuDefaultValue = row['sku']
    effectiveDate = row['effectiveDate']
    termAttributes = row['termAttributes']
    
    # Παίρνουμε το λεξικό του priceDimensions
    priceDimensionDict = row['priceDimensions']

    # Κοιτάζει εάν όντως το priceDimensions είναι λεξικό. Αν δεν είναι δεν μπαίνει καν μέσα στο loop. Έτσι και δεν κρασάρει, και αποφεύγω να δημιουργήσω γραμμές στις οποίες τα δεδομένα είναι ελλιπή
    if isinstance(priceDimensionDict, dict):
        # ΔΌπως και στο πάνω κελί, με τον τρόπο αυτό αγνοούμε το εσωτετικό xxx.xxx.xxx
        for dimensionsDict_hash_key, dimensionsDict_content in priceDimensionDict.items():
            
            # Το pricePerUnit είναι και αυτό με την σειρά του λεξικού οποτε πριν εφαρμόσω την μέθοδο get προσέχω για να βεβαιωθώ ότι το βρήκα και δεν έπεσα στην περίπτωση "κακών" δεδομένων
            price_per_unit_dict = dimensionsDict_content.get('pricePerUnit', {})
            usd_price = price_per_unit_dict.get('USD') if isinstance(price_per_unit_dict, dict) else None
            
            item = {
                'skuNew': skuNew,
                'offerTermCode': offerTermCode,
                'sku': skuDefaultValue,
                'effectiveDate': effectiveDate,
                'termAttributes': termAttributes,
                'rateCode': dimensionsDict_content.get('rateCode'),
                'description': dimensionsDict_content.get('description'),
                'beginRange': dimensionsDict_content.get('beginRange'),
                'endRange': dimensionsDict_content.get('endRange'),
                'unit': dimensionsDict_content.get('unit'),
                'priceUSD': usd_price,   # Aυτό το πεδίο μέσω του ελέγχου που κάναμε πιο πάνω, ή θα είναι None ή θα έχει κάποια τιμή. Οπότε θα το χρησιμοποιήσω μετά για έλεγχω
                'appliesTo': dimensionsDict_content.get('appliesTo')
            }
            itemsList.append(item)

df_terms_final = pd.DataFrame(itemsList)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (9643, 12)


,skuNew,offerTermCode,sku,effectiveDate,termAttributes,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,UBHUUCQA2BAWFNFA,JRTCKXETXF,UBHUUCQA2BAWFNFA,2026-06-01T00:00:00Z,{},UBHUUCQA2BAWFNFA.JRTCKXETXF.6YS6EN2CT7,$0.015 per GB - Asia Pacific (Osaka) Data Tran...,0,Inf,GB,0.0150000000,[]
1,5PP56XDP5UX3QP8N,JRTCKXETXF,5PP56XDP5UX3QP8N,2026-06-01T00:00:00Z,{},5PP56XDP5UX3QP8N.JRTCKXETXF.6YS6EN2CT7,$0.00 per GB - Canada West (Calgary) Data Tran...,0,Inf,GB,0.0000000000,[]
2,7H35Q77CGKE5UBSJ,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,{},7H35Q77CGKE5UBSJ.JRTCKXETXF.PGHJ3S3EYE,$0.025 per GB - first 50 TB / month of storage...,0,51200,GB-Mo,0.0250000000,[]
3,7H35Q77CGKE5UBSJ,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,{},7H35Q77CGKE5UBSJ.JRTCKXETXF.D42MF2PVJS,$0.024 per GB - next 450 TB / month of storage...,51200,512000,GB-Mo,0.0240000000,[]
4,7H35Q77CGKE5UBSJ,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,{},7H35Q77CGKE5UBSJ.JRTCKXETXF.PXJDJ3YRG3,$0.023 per GB - storage used / month over 500 TB,512000,Inf,GB-Mo,0.0230000000,[]


##### Επεξεργασία πίνακα με τα δεδομένα τιμολόγησης 
1. Λίστα appliesTo ή οποία μπορεί να φαίνεται κενή, αλλά θα την φροντίσουμε με explode και reindex για κάθε ενδεχόμενο. Επίσης το το πεδίο termAtrributes το οποίο είναι ένα dict κενό, θα το αφαιρέσουμε καθώς μετά από μελέτη του documentaion κρίθηκε άχρηστο αφ'ης στιγμής κρατάμε μόνο On-Demand εγγραφές Δεν χρειάζονται περίεργα loop ή εντολές σύνθετες. Απλή χρήση των 2 εντολών. 

In [74]:
df_terms_final = df_terms_final.explode('appliesTo').reset_index(drop=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (9643, 12)


,skuNew,offerTermCode,sku,effectiveDate,termAttributes,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,UBHUUCQA2BAWFNFA,JRTCKXETXF,UBHUUCQA2BAWFNFA,2026-06-01T00:00:00Z,{},UBHUUCQA2BAWFNFA.JRTCKXETXF.6YS6EN2CT7,$0.015 per GB - Asia Pacific (Osaka) Data Tran...,0,Inf,GB,0.0150000000,NaN
1,5PP56XDP5UX3QP8N,JRTCKXETXF,5PP56XDP5UX3QP8N,2026-06-01T00:00:00Z,{},5PP56XDP5UX3QP8N.JRTCKXETXF.6YS6EN2CT7,$0.00 per GB - Canada West (Calgary) Data Tran...,0,Inf,GB,0.0000000000,NaN
2,7H35Q77CGKE5UBSJ,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,{},7H35Q77CGKE5UBSJ.JRTCKXETXF.PGHJ3S3EYE,$0.025 per GB - first 50 TB / month of storage...,0,51200,GB-Mo,0.0250000000,NaN
3,7H35Q77CGKE5UBSJ,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,{},7H35Q77CGKE5UBSJ.JRTCKXETXF.D42MF2PVJS,$0.024 per GB - next 450 TB / month of storage...,51200,512000,GB-Mo,0.0240000000,NaN
4,7H35Q77CGKE5UBSJ,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,{},7H35Q77CGKE5UBSJ.JRTCKXETXF.PXJDJ3YRG3,$0.023 per GB - storage used / month over 500 TB,512000,Inf,GB-Mo,0.0230000000,NaN


In [75]:
if 'termAttributes' in df_terms_final.columns:
    df_terms_final = df_terms_final.drop(columns=['termAttributes'])

print (f"Dimension (rows,cols): {df_terms_final.shape}")
df_terms_final.head()


Dimension (rows,cols): (9643, 11)


,skuNew,offerTermCode,sku,effectiveDate,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,UBHUUCQA2BAWFNFA,JRTCKXETXF,UBHUUCQA2BAWFNFA,2026-06-01T00:00:00Z,UBHUUCQA2BAWFNFA.JRTCKXETXF.6YS6EN2CT7,$0.015 per GB - Asia Pacific (Osaka) Data Tran...,0,Inf,GB,0.0150000000,NaN
1,5PP56XDP5UX3QP8N,JRTCKXETXF,5PP56XDP5UX3QP8N,2026-06-01T00:00:00Z,5PP56XDP5UX3QP8N.JRTCKXETXF.6YS6EN2CT7,$0.00 per GB - Canada West (Calgary) Data Tran...,0,Inf,GB,0.0000000000,NaN
2,7H35Q77CGKE5UBSJ,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,7H35Q77CGKE5UBSJ.JRTCKXETXF.PGHJ3S3EYE,$0.025 per GB - first 50 TB / month of storage...,0,51200,GB-Mo,0.0250000000,NaN
3,7H35Q77CGKE5UBSJ,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,7H35Q77CGKE5UBSJ.JRTCKXETXF.D42MF2PVJS,$0.024 per GB - next 450 TB / month of storage...,51200,512000,GB-Mo,0.0240000000,NaN
4,7H35Q77CGKE5UBSJ,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,7H35Q77CGKE5UBSJ.JRTCKXETXF.PXJDJ3YRG3,$0.023 per GB - storage used / month over 500 TB,512000,Inf,GB-Mo,0.0230000000,NaN


2. Αρχικά παρατηρούμε 2 στήλες με το sku της υπηρεσίας. Η μία ήταν εξαρχής μέσα στα δεδομένα τιμολόγησης με την ονομασία sku, και η άλλη προστέθηκε κατά το άνοιγμα των υπηρεσιών. Συγκεκριμένα όλα τα δεδομένα terms είχαν σαν αρχικό αναγνωριστικό το sku χύμα, και μετά τα δεδομένα: κάπως έτσι "xxx: {dict with pricing info}". Οπότε το αρχικό κλειδί το κάναμε στήλη. Τώρα θα γράψουμε κώδικα ο οποίος ελέγχει αν υπάρχει ταύτιση skuNew με sku, αν δεν υπάρχει θα πετάει την εγγραφή, και στο τέλος θα αφαιρεί μία από τις δύο στήλες.

In [76]:
rowCount = len(df_terms_final)

#Βάζω το if για να μπορώ να τρέχω το κελί και μόνο του χωρίς να πετάει error
if 'skuNew' in df_terms_final.columns and 'sku' in df_terms_final.columns:

    # Φτιάχνω την συνθήκη ελέγχου - διαγραφής μιας υπηρεσίας και την εφαρμόζω απευθείας μετά πάνω στο dataframe. Γλιτώνω το loop 
    condition = (df_terms_final['skuNew'] == df_terms_final['sku'])

    df_filtered_terms = df_terms_final[condition]

    df_terms_final = df_filtered_terms.copy()

print(f"Initial records: {rowCount}")
print(f"Rejected records: {rowCount - len(df_terms_final)}")

if 'skuNew' in df_terms_final.columns:
    df_terms_final = df_terms_final.drop(columns=['skuNew'])

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Initial records: 9643
Rejected records: 0
Dimensions (rows,cols): (9643, 10)


,offerTermCode,sku,effectiveDate,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,JRTCKXETXF,UBHUUCQA2BAWFNFA,2026-06-01T00:00:00Z,UBHUUCQA2BAWFNFA.JRTCKXETXF.6YS6EN2CT7,$0.015 per GB - Asia Pacific (Osaka) Data Tran...,0,Inf,GB,0.0150000000,NaN
1,JRTCKXETXF,5PP56XDP5UX3QP8N,2026-06-01T00:00:00Z,5PP56XDP5UX3QP8N.JRTCKXETXF.6YS6EN2CT7,$0.00 per GB - Canada West (Calgary) Data Tran...,0,Inf,GB,0.0000000000,NaN
2,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,7H35Q77CGKE5UBSJ.JRTCKXETXF.PGHJ3S3EYE,$0.025 per GB - first 50 TB / month of storage...,0,51200,GB-Mo,0.0250000000,NaN
3,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,7H35Q77CGKE5UBSJ.JRTCKXETXF.D42MF2PVJS,$0.024 per GB - next 450 TB / month of storage...,51200,512000,GB-Mo,0.0240000000,NaN
4,JRTCKXETXF,7H35Q77CGKE5UBSJ,2026-06-01T00:00:00Z,7H35Q77CGKE5UBSJ.JRTCKXETXF.PXJDJ3YRG3,$0.023 per GB - storage used / month over 500 TB,512000,Inf,GB-Mo,0.0230000000,NaN


3. Η στήλη effectiveDate είναι σε ίδιο μήκος κύματος με τις στήλες που έχει η azure, και η google στα δεδομένα της. Περιγράφει την ημερομηνία και ώρα εκκίνησης της συγκεκριμένης τιμής που υπάρχει για την υπηρεσία. Στην εργασία δεν μας ενδιαφέρει η ιστορικότητα των δεδομέων, οπότε την αφαιρούμε απευθείας με τις αντίστοιχες εντολές.

In [77]:
if 'effectiveDate' in df_terms_final.columns:
    df_terms_final = df_terms_final.drop(columns=['effectiveDate'])

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (9643, 9)


,offerTermCode,sku,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,JRTCKXETXF,UBHUUCQA2BAWFNFA,UBHUUCQA2BAWFNFA.JRTCKXETXF.6YS6EN2CT7,$0.015 per GB - Asia Pacific (Osaka) Data Tran...,0,Inf,GB,0.0150000000,NaN
1,JRTCKXETXF,5PP56XDP5UX3QP8N,5PP56XDP5UX3QP8N.JRTCKXETXF.6YS6EN2CT7,$0.00 per GB - Canada West (Calgary) Data Tran...,0,Inf,GB,0.0000000000,NaN
2,JRTCKXETXF,7H35Q77CGKE5UBSJ,7H35Q77CGKE5UBSJ.JRTCKXETXF.PGHJ3S3EYE,$0.025 per GB - first 50 TB / month of storage...,0,51200,GB-Mo,0.0250000000,NaN
3,JRTCKXETXF,7H35Q77CGKE5UBSJ,7H35Q77CGKE5UBSJ.JRTCKXETXF.D42MF2PVJS,$0.024 per GB - next 450 TB / month of storage...,51200,512000,GB-Mo,0.0240000000,NaN
4,JRTCKXETXF,7H35Q77CGKE5UBSJ,7H35Q77CGKE5UBSJ.JRTCKXETXF.PXJDJ3YRG3,$0.023 per GB - storage used / month over 500 TB,512000,Inf,GB-Mo,0.0230000000,NaN


4. Κλιμακωτές χρεώσεις!! Όλοι οι πάροχοι τις έχουν και σε όλους τις παραλέιπω και κρατάω μόνο την πρώτη βαθμίδα χρέωσης. Το ίδιο και εδω!! Η aws αναπαριστά τις κλιμακωτές χρεώσεις μέσω των πεδίων beginRange και endRange τα οποία καθορίζουν τα όρια. Το 0 στο beginRange είναι αυτό το οποίο μας ενδιαφέρει καθώς είναι η πρώτη - βασική βαθμίδα χρέσωσης. Οπότε θα κρατήσουμε όλες τις εγγραφές εκείνες που έχουν beginRange == 0, και μετά τις στήλες με τα όρια θα τις αφαιρέσουμε, αφού καμία σημασία δεν θα έχουν πλέον.
2 παρατηρήσεις:
- Η aws διατηρεί το ίδιο sku ανάμεσα στις κλίμακες, αλλά αλλάζει το rateCode και συγκεκριμένα τα τελευταία του ψηφία
- Το offerTermCode παρατηρούμε ότι είναι ίδιο για πολλες εγγραφές

In [79]:
tierRates = df_terms_final['beginRange'].unique()
print(tierRates)

['0' '51200' '512000' '25000000000' '100000000000' '1024' '1024000'
 '5120000']


In [ ]:
if 'beginRange' in df_terms_final.columns and 'endRange' in df_terms_final.columns:

    df_terms_final = df_terms_final[df_terms_final['beginRange'] == '0']

    df_terms_final.drop(columns=['beginRange', 'endRange'], inplace=True)

df_terms_final.reset_index(drop=True, inplace=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (9031, 7)


,offerTermCode,sku,rateCode,description,unit,priceUSD,appliesTo
0,JRTCKXETXF,UBHUUCQA2BAWFNFA,UBHUUCQA2BAWFNFA.JRTCKXETXF.6YS6EN2CT7,$0.015 per GB - Asia Pacific (Osaka) Data Tran...,GB,0.0150000000,NaN
1,JRTCKXETXF,5PP56XDP5UX3QP8N,5PP56XDP5UX3QP8N.JRTCKXETXF.6YS6EN2CT7,$0.00 per GB - Canada West (Calgary) Data Tran...,GB,0.0000000000,NaN
2,JRTCKXETXF,7H35Q77CGKE5UBSJ,7H35Q77CGKE5UBSJ.JRTCKXETXF.PGHJ3S3EYE,$0.025 per GB - first 50 TB / month of storage...,GB-Mo,0.0250000000,NaN
3,JRTCKXETXF,EXBKATS7UXZKUK4R,EXBKATS7UXZKUK4R.JRTCKXETXF.6YS6EN2CT7,$0.00 per GB - Asia Pacific (Melbourne) Data T...,GB,0.0000000000,NaN
4,JRTCKXETXF,YD4SUKU86M9QUVVW,YD4SUKU86M9QUVVW.JRTCKXETXF.6YS6EN2CT7,$0.01 per GB fee for bytes processed for sort ...,GB,0.0100000000,NaN
